# ISMIP7 forcing as a DataTree

The ISMIP7 forcing datasets on [source.coop](https://source.coop/ismip) are about
285,000 NetCDF files (6 TB). The [englacial/ismip7](https://source.coop/englacial/ismip7)
Icechunk stores hold *virtual references* into those files (22 GB of metadata, no data
copied). One store per ice sheet, `icechunk-ais` and `icechunk-gris`, laid out as

    <ESM>/<experiment>/<product>/<resolution>m      e.g. CESM2-WACCM/ssp585/SDBN1/2000m

Each resolution group is one xarray Dataset: a normalised monthly (or annual) time axis
in `days since 1850-01-01`, the exact ISMIP7 grid in `x`/`y`, and a `crs` variable with
the EPSG code and WKT. The group above the resolutions (the *family*, e.g.
`CESM2-WACCM/ssp585/SDBN1`) carries an OME-NGFF `multiscales` attribute naming its
levels, so a multiscale-aware reader (gridlook) opens the family as one pyramid.

Reading needs no credentials. Everything below uses the standard tools, `icechunk`,
`zarr` and `xarray`; nothing from `ismip7_helper`.


## Open the store

In [ ]:
import icechunk
import xarray as xr

# the Antarctic store (Greenland: prefix="englacial/ismip7/icechunk-gris")
storage = icechunk.s3_storage(
    bucket="us-west-2.opendata.source.coop",
    prefix="englacial/ismip7/icechunk-ais",
    region="us-west-2",
    anonymous=True,
)
# the store holds references into the ISMIP7 NetCDF files; icechunk follows them
# only into a bucket you have authorised
containers = icechunk.containers_credentials(
    {"s3://us-west-2.opendata.source.coop/ismip/": icechunk.s3_anonymous_credentials()}
)
repo = icechunk.Repository.open(storage, authorize_virtual_chunk_access=containers)
store = repo.readonly_session("main").store

# the series run to 2300, past datetime64[ns]: decode time with cftime
TIME = xr.coders.CFDatetimeCoder(use_cftime=True)
# rustytree opens the whole store in seconds (concurrent walk in Rust); zarr works too
ENGINE = "rustytree" if "rustytree" in xr.backends.list_engines() else "zarr"

## The whole store is one lazy tree

In [ ]:
tree = xr.open_datatree(store, engine=ENGINE, consolidated=False, decode_times=TIME)
list(tree.children)

In [ ]:
# groups are ESM / experiment / family / resolution
family = tree["CESM2-WACCM/historical/SDBN1"]
family

In [ ]:
# the resolutions of a family, finest first (OME-NGFF multiscales)
family.attrs["multiscales"][0]["datasets"]

## A resolution is a Dataset, a variable is a DataArray

In [ ]:
# one resolution is a Dataset
ds8 = family["8000m"].to_dataset()
ds2 = family["2000m"].to_dataset()
ds8

In [ ]:
acabf = ds8["acabf"]
print(acabf.attrs["long_name"], acabf.attrs["units"], "| colour range", acabf.attrs["actual_range"])
print(acabf.time.values[0], "->", acabf.time.values[-1], f"({acabf.sizes['time']} steps)")

## Two resolutions of the same field

In [ ]:
# the same month at 8 km and 2 km: only that step's chunks are read (2 MB and 37 MB)
import matplotlib.pyplot as plt

fig, (ax8, ax2) = plt.subplots(1, 2, figsize=(12, 5.5))
ds8["acabf"].isel(time=0).plot(ax=ax8, x="x", y="y", robust=True)
ds2["acabf"].isel(time=0).plot(ax=ax2, x="x", y="y", robust=True)
ax8.set_title(ds8.attrs["ismip7_grid"])
ax2.set_title(ds2.attrs["ismip7_grid"]);

In [ ]:
# a 400 km window at 2 km, averaged over the first year; nothing is read until .plot
box = ds2["acabf"].isel(time=slice(0, 12), x=slice(1400, 1600), y=slice(1400, 1600))
box.mean("time").plot(x="x", y="y", robust=True, figsize=(5, 5));

## Everything in the store

In [ ]:
# every family in the store, with its resolutions
for node in tree.subtree:
    if "multiscales" in node.attrs:
        print(node.path, [d["path"] for d in node.attrs["multiscales"][0]["datasets"]])

## Greenland

In [ ]:
# Greenland: a second store with the same layout (EPSG:3413 at 4 km and 1 km)
storage_gris = icechunk.s3_storage(
    bucket="us-west-2.opendata.source.coop",
    prefix="englacial/ismip7/icechunk-gris",
    region="us-west-2",
    anonymous=True,
)
repo_gris = icechunk.Repository.open(storage_gris, authorize_virtual_chunk_access=containers)
gris = xr.open_datatree(
    repo_gris.readonly_session("main").store, engine=ENGINE, consolidated=False, decode_times=TIME
)
gris["CESM2-WACCM/historical/SDBN1"]